# V6.5 — Camber YOLO11n@640 Data-Ablation Train

**Bước này mới bắt đầu train.**

Đây chưa phải promotion/final V6.5. Mục tiêu là kiểm tra xem **nguồn dữ liệu mới có thật sự giúp đúng class đang yếu hay không** trước khi tốn compute cho YOLO11m@1024 và RT-DETR.

## Experiment

**Base:** V6.2E immutable  
**+ V6.5 candidate overlay:**
- MPCD train → `broken_strand`
- Power Equipment train → `broken_strand / foreign_object / bird_nest`
- relation-aware synthetic foreign objects → TRAIN only

**VAL:** V6.2E frozen VAL, không thay đổi  
**TEST:** không chạy

**Model:** YOLO11n  
**Resolution:** 640  
**Epochs:** 50  
**Purpose:** same-scale data ablation against old V6.2E YOLO11n@640 baseline.

## Vì sao chưa lấy V6.4 exact train làm base?

Artifact V6.4 hiện không replay được chính xác toàn bộ training runtime (79 GT edits + 600 hard crops). Vì vậy experiment này chỉ trả lời:

> "Các nguồn V6.5 mới có giá trị không?"

Nó **không** được gọi là final V6.5-vs-V6.4 comparison.

Nếu data ablation thắng, bước sau mới freeze canonical V6.5 và chạy YOLO11m@1024 / RT-DETR trên cùng canonical dataset.

In [44]:
# ==============================================================================
# 0. CONFIG + CAMBER AUTH
# ==============================================================================
from pathlib import Path
import os, sys, json, shutil, subprocess, textwrap

STAGE_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage"
COMPOSE_ROOT = f"{STAGE_ROOT}/compose_qa"
JOB_ROOT = f"{STAGE_ROOT}/jobs"
TRAIN_ROOT = "stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation"

# Leave None to auto-discover when only one compose candidate exists.
COMPOSE_TAG_OVERRIDE = None

NODE_SIZE = "MEDIUM"

# Same YOLO11n@640 baseline scale
MODEL = "yolo11n.pt"
IMGSZ = 640
EPOCHS = 50
BATCH = -1
DEVICE = 0
SEED = 20261007

key = os.environ.get("CAMBER_API_KEY")
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("CAMBER_API_KEY")
    except Exception:
        key = None

if not key:
    raise RuntimeError("Bật CAMBER_API_KEY trong Kaggle Secrets.")

os.environ["CAMBER_API_KEY"] = key

camber_bin = Path.home() / ".camber/bin/camber"
if shutil.which("camber") is None and not camber_bin.exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True,
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)
CAMBER = shutil.which("camber") or str(camber_bin)

assert Path(CAMBER).exists()

RAM_TMP = Path("/dev/shm/v65_train_launcher")
RAM_TMP.mkdir(parents=True, exist_ok=True)

print("CAMBER:", CAMBER)
print("COMPOSE_ROOT:", COMPOSE_ROOT)

CAMBER: /root/.camber/bin/camber
COMPOSE_ROOT: stash://bunpmc/projects/yolov11sdi/datasets/v6p5_external_stage/compose_qa


In [45]:
# ==============================================================================
# 1. DISCOVER COMPOSE TAG SERVER-SIDE
# ==============================================================================
def stash_ls_recursive(remote):
    r = subprocess.run(
        [CAMBER, "stash", "ls", "-r", remote],
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout

if COMPOSE_TAG_OVERRIDE:
    COMPOSE_TAG = COMPOSE_TAG_OVERRIDE
else:
    listing = stash_ls_recursive(COMPOSE_ROOT)
    tags = set()

    for line in listing.splitlines():
        if "v6p5_compose_" not in line:
            continue
        # Robustly extract tag token from arbitrary stash-list formatting.
        parts = line.replace("\\","/").split("/")
        for part in parts:
            if part.startswith("v6p5_compose_"):
                tags.add(part.strip())

    tags = sorted(tags)

    print("Compose candidates:", tags)

    if len(tags) == 0:
        raise RuntimeError(
            "Không tìm thấy compose candidate. "
            "Chạy KAGGLE_V6P5_CAMBER_COMPOSE_QA_LAUNCHER trước."
        )
    elif len(tags) == 1:
        COMPOSE_TAG = tags[0]
    else:
        raise RuntimeError(
            "Có nhiều compose candidate. "
            "Set COMPOSE_TAG_OVERRIDE ở cell CONFIG bằng tag muốn train: "
            + ", ".join(tags)
        )

print("SELECTED COMPOSE_TAG:", COMPOSE_TAG)
COMPOSE_REMOTE = f"{COMPOSE_ROOT}/{COMPOSE_TAG}"

Compose candidates: []


RuntimeError: Không tìm thấy compose candidate. Chạy KAGGLE_V6P5_CAMBER_COMPOSE_QA_LAUNCHER trước.

In [ ]:
# ==============================================================================
# 2. CAMBER GPU TRAIN WORKER
# ==============================================================================
TRAIN_WORKER_TEMPLATE = r"""
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, tarfile, io
import pandas as pd
import numpy as np

COMPOSE_TAG = __COMPOSE_TAG__
COMPOSE_REMOTE = __COMPOSE_REMOTE__
TRAIN_ROOT = __TRAIN_ROOT__
STAGE_ROOT = __STAGE_ROOT__

V6ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
V6TAG = "v6p2e_23562a45b343"

MODEL = __MODEL__
IMGSZ = __IMGSZ__
EPOCHS = __EPOCHS__
BATCH = __BATCH__
DEVICE = __DEVICE__
SEED = __SEED__

ROOT = Path("/tmp/v65_train_worker")
CACHE = ROOT/"cache"
DATASET = ROOT/"dataset"
RUNS = ROOT/"runs"

for p in [CACHE,DATASET,RUNS]:
    p.mkdir(parents=True,exist_ok=True)

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "ultralytics==8.4.172","pandas","numpy","pyyaml","pillow"
])

import yaml
from ultralytics import YOLO

def ensure_camber():
    c=shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    if not Path(c).exists():
        subprocess.check_call(
            "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
            shell=True,
        )
        os.environ["PATH"]=f"{Path.home()}/.camber/bin:"+os.environ.get("PATH","")
        c=shutil.which("camber") or str(Path.home()/".camber/bin/camber")
    assert Path(c).exists()
    return c

CAMBER=ensure_camber()

def run(cmd):
    r=subprocess.run(cmd,text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0:
        raise RuntimeError((r.stderr or r.stdout)[:10000])
    return r

def get(remote,local):
    local=Path(local)
    local.parent.mkdir(parents=True,exist_ok=True)
    if not local.exists():
        run([CAMBER,"stash","cp",remote,str(local)])
    return local

def put(local,remote):
    run([CAMBER,"stash","cp",str(local),remote])

def norm_names(names):
    if isinstance(names,dict):
        return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names or [])

def safe_link_or_copy(src,dst):
    src=Path(src); dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src,dst)
    except Exception:
        shutil.copy2(src,dst)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        while True:
            b=f.read(1024*1024)
            if not b: break
            h.update(b)
    return h.hexdigest()

# =============================================================================
# A. RECONSTRUCT V6.2E BASE
# =============================================================================
freeze_path=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e.freeze.json",
    CACHE/"v6p2e.freeze.json",
)
freeze=json.loads(freeze_path.read_text())

base_tar=get(
    f"{V6ROOT}/curated_v6_rs1280_{freeze['base_tag']}.tar",
    CACHE/"base.tar",
)
overlay_tar=get(
    f"{V6ROOT}/v6p1/{freeze['parent_dataset_tag']}/v6p1_fotl_overlay_{freeze['overlay_tag']}.tar.gz",
    CACHE/"overlay.tar.gz",
)
labels_zip=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e_reconciled_labels.zip",
    CACHE/"labels.zip",
)
manifest_csv=get(
    f"{V6ROOT}/v6p2e/{V6TAG}/v6p2e_split_manifest.csv",
    CACHE/"manifest.csv",
)

BASE_EX=CACHE/"base_ex"
OV_EX=CACHE/"overlay_ex"
RECON=CACHE/"recon"
for p in [BASE_EX,OV_EX,RECON]:
    p.mkdir(parents=True,exist_ok=True)

def find_sixclass_root(root):
    target=[
        "insulator",
        "insulator_broken",
        "insulator_flashover",
        "bird_nest",
        "foreign_object",
        "broken_strand",
    ]
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names"))==target:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"Six-class root not found under {root}")

if not any(BASE_EX.rglob("data.yaml")):
    with tarfile.open(base_tar,"r:*") as t:
        t.extractall(BASE_EX,filter="data")

if not any(OV_EX.rglob("data.yaml")):
    with tarfile.open(overlay_tar,"r:*") as t:
        t.extractall(OV_EX,filter="data")

if not any(RECON.rglob("*.txt")):
    with zipfile.ZipFile(labels_zip) as z:
        z.extractall(RECON)

BASE_ROOT=find_sixclass_root(BASE_EX)
OV_ROOT=find_sixclass_root(OV_EX)
manifest=pd.read_csv(manifest_csv)

def locate(row):
    old=str(row["image_path_source"]).replace("\\","/")
    oldsplit=str(row["previous_v6p1_split"])
    marker=f"/images/{oldsplit}/"
    suffix=old.split(marker,1)[1]

    if suffix.startswith("base/"):
        root=BASE_ROOT; rel=suffix[len("base/"):]
    elif suffix.startswith("fotl/"):
        root=OV_ROOT; rel=suffix[len("fotl/"):]
    else:
        raise RuntimeError(suffix)

    return (
        root/"images"/oldsplit/rel,
        root/"labels"/oldsplit/Path(rel).with_suffix(".txt"),
    )

def label_for(row,lp0):
    rp=RECON/f"{row['sha256']}.txt"
    if str(row.get("resolution","")).startswith("merged") and rp.exists():
        return rp
    return lp0

for s in ["train","val"]:
    (DATASET/"images"/s).mkdir(parents=True,exist_ok=True)
    (DATASET/"labels"/s).mkdir(parents=True,exist_ok=True)

unresolved=[]

for i,row in manifest[manifest["split"].isin(["train","val"])].iterrows():
    split=str(row["split"])
    ip,lp0=locate(row)
    lp=label_for(row,lp0)

    if not ip.exists() or not lp.exists():
        unresolved.append((int(i),str(ip),str(lp)))
        continue

    name=f"{str(row['sha256'])[:16]}_{i:06d}{ip.suffix.lower()}"
    safe_link_or_copy(ip,DATASET/"images"/split/name)
    safe_link_or_copy(lp,DATASET/"labels"/split/Path(name).with_suffix(".txt"))

if unresolved:
    pd.DataFrame(unresolved,columns=["row","image","label"]).to_csv(
        ROOT/"unresolved.csv",index=False
    )
    raise RuntimeError(f"V6.2E unresolved files: {len(unresolved)}")

base_train_count=len(list((DATASET/"images/train").glob("*")))
base_val_count=len(list((DATASET/"images/val").glob("*")))

# =============================================================================
# B. ADD DIRECT V6.5 TRAIN OVERLAYS
# =============================================================================
overlay_sources=[
    (
        f"{STAGE_ROOT}/mpcd/mpcd_broken_strand_stage.tar.gz",
        CACHE/"mpcd.tar.gz",
        "mpcd_broken_strand",
    ),
    (
        f"{STAGE_ROOT}/power_equipment/power_equipment_stage.tar.gz",
        CACHE/"pe.tar.gz",
        "power_equipment_stage",
    ),
]

added_direct=0

for remote,local,rootname in overlay_sources:
    get(remote,local)

    with tarfile.open(local,"r:gz") as t:
        # Stream only train images/labels to avoid materializing external_eval.
        members={m.name:m for m in t.getmembers() if m.isfile()}

        image_members=[
            n for n in members
            if f"{rootname}/images/train/" in n
        ]

        for n in image_members:
            m=members[n]
            raw=t.extractfile(m).read()

            basename=Path(n).name
            source_prefix="mpcd" if "mpcd" in rootname else "pe"
            name=f"v65_{source_prefix}_{basename}"

            di=DATASET/"images/train"/name
            di.write_bytes(raw)

            label_name=str(
                Path(n.replace("/images/train/","/labels/train/"))
                .with_suffix(".txt")
            )

            lm=members.get(label_name)
            if lm is None:
                raise RuntimeError(f"Missing overlay label: {label_name}")

            dl=DATASET/"labels/train"/Path(name).with_suffix(".txt")
            dl.write_bytes(t.extractfile(lm).read())
            added_direct+=1

    local.unlink(missing_ok=True)

# =============================================================================
# C. ADD SYNTHETIC V6.5 TRAIN OVERLAY
# =============================================================================
# Discover exact synthetic archive filename from compose index.
index_remote=f"{COMPOSE_REMOTE}/{COMPOSE_TAG}.index.json"
index_local=get(index_remote,CACHE/"compose.index.json")
compose_index=json.loads(index_local.read_text())

synthetic_remote=compose_index["synthetic_archive"]
synthetic_local=get(synthetic_remote,CACHE/"synthetic.tar.gz")

added_synth=0

with tarfile.open(synthetic_local,"r:gz") as t:
    members={m.name:m for m in t.getmembers() if m.isfile()}

    image_members=[
        n for n in members
        if "synthetic_foreign/images/train/" in n
    ]

    for n in image_members:
        m=members[n]
        basename=Path(n).name
        name=f"v65_syn_{basename}"

        (DATASET/"images/train"/name).write_bytes(
            t.extractfile(m).read()
        )

        label_name=str(
            Path(n.replace("/images/train/","/labels/train/"))
            .with_suffix(".txt")
        )

        lm=members.get(label_name)
        if lm is None:
            raise RuntimeError(f"Missing synthetic label: {label_name}")

        (DATASET/"labels/train"/Path(name).with_suffix(".txt")).write_bytes(
            t.extractfile(lm).read()
        )

        added_synth+=1

synthetic_local.unlink(missing_ok=True)

# =============================================================================
# D. WRITE DATA YAML
# =============================================================================
names=[
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

data_yaml=DATASET/"data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path":str(DATASET),
    "train":"images/train",
    "val":"images/val",
    "names":{i:n for i,n in enumerate(names)},
},sort_keys=False))

train_count=len(list((DATASET/"images/train").glob("*")))
val_count=len(list((DATASET/"images/val").glob("*")))

# =============================================================================
# E. TRAIN SAME-SCALE YOLO11n@640 / 50 epochs
# =============================================================================
model=YOLO(MODEL)

result=model.train(
    data=str(data_yaml),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    device=DEVICE,
    workers=8,
    pretrained=True,
    seed=SEED,
    project=str(RUNS),
    name="yolo11n_v6p5_data_ablation_640",
    exist_ok=True,
    plots=False,
    verbose=True,
)

run_dir=RUNS/"yolo11n_v6p5_data_ablation_640"
best=run_dir/"weights/best.pt"
assert best.exists(), best

# =============================================================================
# F. STRICT VAL EVALUATION
# =============================================================================
best_model=YOLO(str(best))

metrics=best_model.val(
    data=str(data_yaml),
    split="val",
    imgsz=IMGSZ,
    batch=16,
    device=DEVICE,
    plots=False,
    verbose=False,
)

m=metrics.box
all_ap=np.asarray(m.all_ap,dtype=float)
ap50=np.asarray(m.ap50,dtype=float)
class_ids=[int(x) for x in np.asarray(m.ap_class_index).tolist()]

if all_ap.ndim==1:
    all_ap=all_ap[:,None]

thresholds=np.arange(0.50,0.50+0.05*all_ap.shape[1],0.05)

def col_for(t):
    return int(np.argmin(np.abs(thresholds-t)))

per_class=[]

for ri,cid in enumerate(class_ids):
    vals=all_ap[ri]
    row={
        "cid":cid,
        "class":names[cid],
        "AP50":float(ap50[ri]),
        "AP50_95":float(np.mean(vals)),
        "gap":float(ap50[ri]-np.mean(vals)),
        "AP75":float(vals[col_for(0.75)]) if all_ap.shape[1]>1 else None,
        "AP90":float(vals[col_for(0.90)]) if all_ap.shape[1]>1 else None,
        "AP95":float(vals[col_for(0.95)]) if all_ap.shape[1]>1 else None,
    }
    per_class.append(row)

# Old V6.2E YOLO11n@640 reference
baseline={
    "mAP50":0.768,
    "mAP50_95":0.413,
    "gap":0.355,
    "per_class":{
        "insulator":{"AP50":0.934,"AP50_95":0.638},
        "insulator_broken":{"AP50":0.800,"AP50_95":0.407},
        "insulator_flashover":{"AP50":0.755,"AP50_95":0.326},
        "bird_nest":{"AP50":0.831,"AP50_95":0.455},
        "foreign_object":{"AP50":0.537,"AP50_95":0.295},
        "broken_strand":{"AP50":0.749,"AP50_95":0.356},
    },
}

report={
    "experiment":"V6.2E + V6.5 external candidate data ablation",
    "compose_tag":COMPOSE_TAG,
    "model":"YOLO11n",
    "imgsz":IMGSZ,
    "epochs":EPOCHS,
    "base_train_images":base_train_count,
    "added_direct_train_images":added_direct,
    "added_synthetic_train_images":added_synth,
    "final_train_images":train_count,
    "val_images":val_count,
    "mAP50":float(m.map50),
    "mAP50_95":float(m.map),
    "gap":float(m.map50-m.map),
    "precision":float(m.mp),
    "recall":float(m.mr),
    "per_class":per_class,
    "reference_v6p2e_yolo11n":baseline,
}

report["delta_vs_v6p2e"]={
    "mAP50":report["mAP50"]-baseline["mAP50"],
    "mAP50_95":report["mAP50_95"]-baseline["mAP50_95"],
    "gap":report["gap"]-baseline["gap"],
}

for row in report["per_class"]:
    b=baseline["per_class"].get(row["class"])
    if b:
        row["delta_AP50_vs_v6p2e"]=row["AP50"]-b["AP50"]
        row["delta_AP50_95_vs_v6p2e"]=row["AP50_95"]-b["AP50_95"]

report_path=ROOT/"v6p5_yolo11n_data_ablation_report.json"
report_path.write_text(json.dumps(report,indent=2))

pd.DataFrame(per_class).to_csv(
    ROOT/"v6p5_yolo11n_strict_iou_per_class.csv",
    index=False,
)

# Practical promotion signal for the NEW sources:
# foreign AP50/AP50:95 should improve, strand strict metric should improve,
# and global AP50:95 must not materially regress.
pc={r["class"]:r for r in per_class}

promotion={
    "global_map5095_not_regress_gt_1pt":
        report["mAP50_95"] >= baseline["mAP50_95"]-0.01,
    "foreign_AP50_improves_2pt":
        pc.get("foreign_object",{}).get("AP50",0) >= baseline["per_class"]["foreign_object"]["AP50"]+0.02,
    "foreign_AP5095_improves_1pt":
        pc.get("foreign_object",{}).get("AP50_95",0) >= baseline["per_class"]["foreign_object"]["AP50_95"]+0.01,
    "strand_AP5095_improves_1pt":
        pc.get("broken_strand",{}).get("AP50_95",0) >= baseline["per_class"]["broken_strand"]["AP50_95"]+0.01,
}

promotion["candidate_pass"]=all(promotion.values())
report["promotion_gate"]=promotion
report_path.write_text(json.dumps(report,indent=2))

# =============================================================================
# G. PACKAGE + UPLOAD TO CAMBER
# =============================================================================
tag_input=json.dumps({
    "compose_tag":COMPOSE_TAG,
    "model":"YOLO11n",
    "imgsz":IMGSZ,
    "epochs":EPOCHS,
    "train_count":train_count,
},sort_keys=True).encode()

tag="v6p5_y11n_ablation_"+hashlib.sha256(tag_input).hexdigest()[:12]
remote_root=f"{TRAIN_ROOT}/{tag}"

put(best,f"{remote_root}/best.pt")
put(report_path,f"{remote_root}/{report_path.name}")
put(ROOT/"v6p5_yolo11n_strict_iou_per_class.csv",
    f"{remote_root}/v6p5_yolo11n_strict_iou_per_class.csv")

results_csv=run_dir/"results.csv"
if results_csv.exists():
    put(results_csv,f"{remote_root}/results.csv")

done=ROOT/"done.json"
done.write_text(json.dumps({
    "tag":tag,
    "remote_root":remote_root,
    "promotion_gate":promotion,
},indent=2))
put(done,f"{remote_root}/done.json")

print(json.dumps(report,indent=2))
print("REMOTE:",remote_root)

shutil.rmtree(ROOT,ignore_errors=True)
"""

TRAIN_WORKER = (
    TRAIN_WORKER_TEMPLATE
    .replace("__COMPOSE_TAG__", repr(COMPOSE_TAG))
    .replace("__COMPOSE_REMOTE__", repr(COMPOSE_REMOTE))
    .replace("__TRAIN_ROOT__", repr(TRAIN_ROOT))
    .replace("__STAGE_ROOT__", repr(STAGE_ROOT))
    .replace("__MODEL__", repr(MODEL))
    .replace("__IMGSZ__", str(IMGSZ))
    .replace("__EPOCHS__", str(EPOCHS))
    .replace("__BATCH__", str(BATCH))
    .replace("__DEVICE__", str(DEVICE))
    .replace("__SEED__", str(SEED))
)

print("Train worker prepared:", len(TRAIN_WORKER), "chars")

In [ ]:
# ==============================================================================
# 3. WRITE TO RAM + UPLOAD WORKER
# ==============================================================================
worker_py = RAM_TMP / "v65_yolo11n_data_ablation_worker.py"

compile(TRAIN_WORKER, str(worker_py), "exec")
worker_py.write_text(TRAIN_WORKER)

REMOTE_WORKER = f"{JOB_ROOT}/{worker_py.name}"

r = subprocess.run(
    [CAMBER, "stash", "cp", str(worker_py), REMOTE_WORKER],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)

if r.returncode != 0:
    raise RuntimeError(r.stderr or r.stdout)

worker_py.unlink(missing_ok=True)

print("Worker uploaded:")
print(REMOTE_WORKER)

In [ ]:
# ==============================================================================
# 4. SUBMIT CAMBER GPU JOB
# ==============================================================================
command = (
    "bash -lc 'mkdir -p /tmp/v65train && "
    f"camber stash cp {REMOTE_WORKER} /tmp/v65train/worker.py && "
    "python /tmp/v65train/worker.py'"
)

try:
    import camber

    if not hasattr(camber, "mpi"):
        raise ImportError("camber.mpi unavailable")

    job = camber.mpi.create_job(
        command=command,
        node_size=NODE_SIZE,
        num_nodes=1,
        with_gpu=True,
    )

    print("Submitted Camber GPU training job:")
    print(job)

except Exception as e:
    print("Automatic submit unavailable:", repr(e))
    print()
    print("Worker đã nằm trên Camber Stash.")
    print("Trong Camber Nova, chạy:")
    print()
    print(
        f"Run {REMOTE_WORKER} as a Python script on one {NODE_SIZE} GPU node. "
        "This is the YOLO11n@640 50-epoch V6.5 data-ablation experiment."
    )

print("\nCommand:")
print(command)

In [ ]:
# ==============================================================================
# 5. CHECK TRAINING OUTPUTS LATER
# ==============================================================================
print("Checkpoint root:")
print(TRAIN_ROOT)

r = subprocess.run(
    [CAMBER, "stash", "ls", "-r", TRAIN_ROOT],
    text=True,
    capture_output=True,
    env=os.environ.copy(),
)

print((r.stdout or r.stderr)[:20000])

## Cách đọc kết quả

Đây là **data ablation**, nên compare với YOLO11n@640 V6.2E cũ:

- global `mAP50`: 0.768
- global `mAP50:95`: 0.413
- gap: 0.355

Đặc biệt:

- `foreign_object`: 0.537 / 0.295
- `broken_strand`: 0.749 / 0.356

### Gate để đi tiếp YOLO11m@1024

Candidate được xem là đáng giữ nếu:

- global mAP50:95 không giảm quá 1 điểm;
- foreign AP50 tăng ≥ 2 điểm;
- foreign AP50:95 tăng ≥ 1 điểm;
- broken_strand AP50:95 tăng ≥ 1 điểm.

Nếu gate pass:

`review/promotion -> canonical V6.5 -> YOLO11m@1024 40ep -> RT-DETR B0`

Nếu foreign tăng nhưng strand giảm, tách ablation nguồn:
- only foreign expansion
- only broken-strand direct sources

Không cần cưới cả hai source chỉ vì chúng cùng xuất hiện trong một notebook. Dữ liệu cũng nên có quyền chia tay.